In [0]:
%run ../create-secret

# Execução da pipeline sample_mflix -> Bronze
Notebook de desenvolvimento/evidência: chama `ingestion_job.run()` e
`bronze_job.run()` diretamente em sequência, sem depender de um
Databricks Job configurado. Em produção, os dois scripts rodam como
tasks encadeadas — ver `config/databricks_job.yml`.

Este notebook produz as **3 evidências de execução** exigidas em
`SEND_WORK.md`: carga full inicial, incremental sem novidades e
incremental com dados novos.

In [0]:
import sys
sys.path.append("../jobs")
sys.path.append("../src")

import ingestion_job
import bronze_job

PIPELINE_CONFIG = "../config/pipeline_config.yaml"
COLLECTIONS_CONFIG = "../config/collections.json"

mongo_uri = dbutils.secrets.get(scope="conn-db", key="cnn-mongodb-sampleflix")

## Execução 1 — carga full inicial (todas as 6 coleções)

In [0]:
ingestion_id_1, ingest_summaries_1 = ingestion_job.run(
    spark, mongo_uri, PIPELINE_CONFIG, COLLECTIONS_CONFIG
)
for s in ingest_summaries_1:
    print(f"[{s.status}] {s.collection:<20} lidos={s.qtd_lida_origem:<8} watermark_final={s.watermark_final}")

In [0]:
bronze_summaries_1 = bronze_job.run(spark, PIPELINE_CONFIG, COLLECTIONS_CONFIG, ingestion_id_1)
for s in bronze_summaries_1:
    print(f"[{s.status}] {s.collection:<20} bronze={s.qtd_gravada_bronze:<8} quarentena={s.qtd_quarentena}")

In [0]:
display(
    spark.table("meu_catalog.bronze.control_ingestion_log")
    .filter(f"_ingestion_id = '{ingestion_id_1}'")
    .orderBy("stage", "collection")
)
# Print desta célula -> docs/evidencias/execucao_01_full_load.png

## Execução 2 — incremental, sem novidades
Rode esta célula logo em seguida, **sem** inserir dados novos no
Mongo. `qtd_lida_origem` deve vir 0 para movies e comments.

In [0]:
ingestion_id_2, ingest_summaries_2 = ingestion_job.run(
    spark, mongo_uri, PIPELINE_CONFIG, COLLECTIONS_CONFIG,
    only_collections=["movies", "comments"],
)
bronze_job.run(spark, PIPELINE_CONFIG, COLLECTIONS_CONFIG, ingestion_id_2, only_collections=["movies", "comments"])

display(
    spark.table("meu_catalog.bronze.control_ingestion_log")
    .filter(f"_ingestion_id = '{ingestion_id_2}'")
    .orderBy("stage", "collection")
)
# Print desta célula -> docs/evidencias/execucao_02_incremental_sem_novidades.png

## Execução 3 — incremental com dados novos
Antes de rodar a célula abaixo, insira manualmente 1–3 comentários
novos no MongoDB com `date` posterior ao watermark atual, via
mongosh/Compass, por exemplo:
```javascript
db.comments.insertOne({
  name: "Evidencia Teste",
  email: "evidencia@teste.com",
  movie_id: ObjectId("573a1390f29313caabcd4135"),
  text: "Comentario inserido para evidencia de carga incremental.",
  date: new Date()
})
```

In [0]:
ingestion_id_3, ingest_summaries_3 = ingestion_job.run(
    spark, mongo_uri, PIPELINE_CONFIG, COLLECTIONS_CONFIG,
    only_collections=["comments"],
)
bronze_job.run(spark, PIPELINE_CONFIG, COLLECTIONS_CONFIG, ingestion_id_3, only_collections=["comments"])

display(
    spark.table("meu_catalog.bronze.control_ingestion_log")
    .filter(f"_ingestion_id = '{ingestion_id_3}'")
    .orderBy("stage", "collection")
)
# Print desta célula -> docs/evidencias/execucao_03_incremental_com_dados.png

## Prova de idempotência — rodar a execução 3 de novo não duplica

In [0]:
ingestion_id_4, _ = ingestion_job.run(
    spark, mongo_uri, PIPELINE_CONFIG, COLLECTIONS_CONFIG, only_collections=["comments"],
)
bronze_job.run(spark, PIPELINE_CONFIG, COLLECTIONS_CONFIG, ingestion_id_4, only_collections=["comments"])

dup = (
    spark.table("meu_catalog.bronze.sample_mflix__comments")
    .groupBy("_source_id").count().filter("count > 1").count()
)
print(f"_source_id duplicados em bronze.sample_mflix__comments: {dup}")  # deve ser 0